In [11]:
import pandas as pd
import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix, f1_score
import joblib
from scipy import sparse

# ==========================================
# 1. LOAD THE DATA
# ==========================================
print("Loading CSV files...")
train_df = pd.read_csv('/kaggle/input/datasets/varunvijay007/ljp-training/train_split.csv')
val_df = pd.read_csv('/kaggle/input/datasets/varunvijay007/ljp-training/val_split.csv')
test_df = pd.read_csv('/kaggle/input/datasets/varunvijay007/ljp-training/test_split.csv')

# ==========================================
# 2. CLEAN THE TEXT
# ==========================================
print("Cleaning text (This might take a minute)...")

def clean_text(text) -> str:
    # Handle if the text is accidentally stored as a list
    if isinstance(text, list):
        text = ' '.join(text)
    # Handle empty rows
    if not isinstance(text, str) or len(text.strip()) == 0:
        return ""
    
    # Remove common Supreme Court headers and case numbers
    text = re.sub(r'(IN THE SUPREME COURT OF INDIA.*?)\n', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(CIVIL APPELLATE JURISDICTION|CRIMINAL APPELLATE JURISDICTION)', '', text, flags=re.IGNORECASE)
    text = re.sub(r'(Civil|Criminal|Writ)\s+(Appeal|Petition)\s+No\.?\s*\d+.*?\d{4}', '', text)
    
    # Normalize whitespace and remove page numbers
    text = re.sub(r'\n+', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    text = re.sub(r'-\s*\d+\s*-', '', text)
    text = re.sub(r'Page\s+\d+', '', text, flags=re.IGNORECASE)
    
    return text.strip()

# Apply the cleaning function to create the 'clean_text' column
train_df['clean_text'] = train_df['text'].apply(clean_text)
val_df['clean_text'] = val_df['text'].apply(clean_text)
test_df['clean_text'] = test_df['text'].apply(clean_text)

# Ensure no empty strings crash the vectorizer
train_df['clean_text'] = train_df['clean_text'].fillna('')
val_df['clean_text'] = val_df['clean_text'].fillna('')
test_df['clean_text'] = test_df['clean_text'].fillna('')

y_train = train_df['label'].values
y_val   = val_df['label'].values
y_test  = test_df['label'].values

# ==========================================
# 3. BUILD TF-IDF
# ==========================================
print("\nBuilding TF-IDF features...")
MAX_CHARS = 5000

tfidf = TfidfVectorizer(
    max_features=30000,
    ngram_range=(1, 2),
    sublinear_tf=True,
    min_df=3,
    max_df=0.90,
    strip_accents='unicode',
    stop_words='english',
    token_pattern=r'\b[a-zA-Z][a-zA-Z]+\b'
)

# Fit ONLY on train — never on val or test
X_train = tfidf.fit_transform(train_df['clean_text'].astype(str).str[:MAX_CHARS])
X_val  = tfidf.transform(val_df['clean_text'].astype(str).str[:MAX_CHARS])
X_test = tfidf.transform(test_df['clean_text'].astype(str).str[:MAX_CHARS])

print(f"TF-IDF Shape — Train: {X_train.shape}")
print(f"TF-IDF Shape — Test : {X_test.shape}")

# ==========================================
# 4. TRAIN LOGISTIC REGRESSION
# ==========================================
print("\n" + "="*60)
print("TRAINING LOGISTIC REGRESSION")
print("="*60)

lr_model = LogisticRegression(
    max_iter=1000,
    class_weight='balanced',    
    random_state=42,
    C=1.0,
    solver='lbfgs'
)
lr_model.fit(X_train, y_train)

# Validation
lr_val_preds = lr_model.predict(X_val)
print(f"LR Validation Macro F1: {f1_score(y_val, lr_val_preds, average='macro'):.4f}")

# Test
lr_test_preds = lr_model.predict(X_test)
print("\n--- Test Set Results (Logistic Regression) ---")
print(classification_report(y_test, lr_test_preds))
print(f"Accuracy : {accuracy_score(y_test, lr_test_preds):.4f}")
print(f"Macro F1 : {f1_score(y_test, lr_test_preds, average='macro'):.4f}")

# ==========================================
# 5. TRAIN SUPPORT VECTOR MACHINE (SVM)
# ==========================================
print("\n" + "="*60)
print("TRAINING SVM CLASSIFIER (LinearSVC)")
print("="*60)

svm_model = LinearSVC(
    class_weight='balanced',
    random_state=42,
    max_iter=5000,
    C=1.0
)
svm_model.fit(X_train, y_train)

# Validation
svm_val_preds = svm_model.predict(X_val)
print(f"SVM Validation Macro F1: {f1_score(y_val, svm_val_preds, average='macro'):.4f}")

# Test
svm_test_preds = svm_model.predict(X_test)
print("\n--- Test Set Results (SVM) ---")
print(classification_report(y_test, svm_test_preds))
print(f"Accuracy : {accuracy_score(y_test, svm_test_preds):.4f}")
print(f"Macro F1 : {f1_score(y_test, svm_test_preds, average='macro'):.4f}")

# ==========================================
# 6. SAVE EVERYTHING FOR FUTURE USE
# ==========================================
print("\nSaving Models and Matrices to /kaggle/working/ ...")
joblib.dump(tfidf, '/kaggle/working/tfidf_vectorizer.joblib')
joblib.dump(lr_model, '/kaggle/working/lr_model.joblib')
joblib.dump(svm_model, '/kaggle/working/svm_model.joblib')

sparse.save_npz('/kaggle/working/X_train_tfidf.npz', X_train)
sparse.save_npz('/kaggle/working/X_val_tfidf.npz', X_val)
sparse.save_npz('/kaggle/working/X_test_tfidf.npz', X_test)

print("Pipeline Complete! All files saved successfully.")


Loading CSV files...
Cleaning text (This might take a minute)...

Building TF-IDF features...
TF-IDF Shape — Train: (6111, 30000)
TF-IDF Shape — Test : (1071, 30000)

TRAINING LOGISTIC REGRESSION
LR Validation Macro F1: 0.6155

--- Test Set Results (Logistic Regression) ---
              precision    recall  f1-score   support

           0       0.50      0.59      0.54       480
           1       0.61      0.52      0.56       591

    accuracy                           0.55      1071
   macro avg       0.55      0.55      0.55      1071
weighted avg       0.56      0.55      0.55      1071

Accuracy : 0.5490
Macro F1 : 0.5488

TRAINING SVM CLASSIFIER (LinearSVC)
SVM Validation Macro F1: 0.5802

--- Test Set Results (SVM) ---
              precision    recall  f1-score   support

           0       0.48      0.62      0.54       480
           1       0.59      0.45      0.51       591

    accuracy                           0.53      1071
   macro avg       0.53      0.53      0.53

In [8]:
import joblib
from scipy import sparse

# 1. Save the TF-IDF Vectorizer
joblib.dump(tfidf, '/kaggle/working/tfidf_vectorizer.joblib')
print("TF-IDF Vectorizer saved successfully!")

# 2. PRO-TIP: Save your transformed matrices too!
# Since TF-IDF outputs "sparse matrices", we save them using scipy
sparse.save_npz('/kaggle/working/X_train_tfidf.npz', X_train)
sparse.save_npz('/kaggle/working/X_val_tfidf.npz', X_val)
sparse.save_npz('/kaggle/working/X_test_tfidf.npz', X_test)
print("TF-IDF Matrices saved successfully!")

TF-IDF Vectorizer saved successfully!
TF-IDF Matrices saved successfully!
